In [ ]:
import os
import json
from typing import Dict, Any, List
from openai import OpenAI
from tqdm.std import tqdm

In [ ]:
model_html = "the path to the generated HTML files from model"
gold_html = "the path to the gold HTML files with entity tag"
total_samples = 500

model_name = "model's name"
case_result = "output specific case result"

In [ ]:
OPEN_AI_KEY = "your open AI api key"

In [ ]:
Client = OpenAI(api_key=OPEN_AI_KEY)

In [ ]:
JUDGE_PROMPT_TEMPLATE = """ 
#Instruction: You are an expert OCR results inspector for financial documents. You will be judging the quality of model generated HTML results based on image inputs. 
You will be given:
1. Ground truth HTML that contains special entity tags that label information that are financially critical, including 5 types: Number, Temporal, Monetary Unit, Reporting Entity, and Financial Concepts, each entity are wrapped with special spans:
*Number: <number>…</number> 
*Temporal: <temporal>…</temporal>
*Monetary Unit: <monetaryunit>…</monetaryunit>
*Reporting Entity: <reportingentity>…</reportingentity>
*Financial Concepts: <financialconcepts>…</financialconcepts>

    2. Model-generated HTML that was produced from the same image but does not contain those tags.

Your goal is to judge the quality of the financially critical information by comparing it with the ground truth HTML.

Follow all steps carefully and return one JSON object as the final result.

# Step 1: Extract and match entities
    From the ground truth HTML, extract all entities explicitly enclosed by the special tags:
* <number>…</number>  → entity type = "Number"
* <temporal>…</temporal> → entity type = "Temporal"
* <monetaryunit>…</monetaryunit> → entity type = "Monetary Unit"
* <reportingentity>…</reportingentity> → entity type = "Reporting Entity"
* <financialconcepts>…</financialconcepts> → entity type = "Financial Concepts"

Do not infer entities by meaning; only extract those wrapped by these tags. Each entity record must include:
* type - "Number" ,”Temporal”,”Monetary Unit”,”Reporting Entity”,”Financial Concepts”
* value - exact inner text between the tags
* context_hint - short fragment of surrounding text that helps locate it
    Example:
        [
          {"type": "Number", "value": "50,000", "context_hint": "sales charge discounts"},
          {"type": "Temporal", "value": "December 31, 2024", "context_hint": "fiscal year ended"}
        ]
Then, in the generated HTML (which lacks tags), locate each entity’s value within a similar textual or positional context. Count a match as correct only if the same text appears in the correct or very similar paragraph, sentence, or table cell.
    Compute and report:
* total_entities = count of GT entities
* total_entities_with_Number_type = count of GT Number entities
* total_entities_with_Temporal_type = count of GT Temporal entities
* total_entities_with_Monetary_Unit_type = count of GT Monetary Unit entities
* total_entities_with_Reporting_Entity_type = count of GT Reporting Entity entities
* total_entities_with_Financial_Concepts_type = count of GT Financial Concepts entities
* correct_entities = the number of entities that is correctly found
* correct_entities_with_Number_type = the number of entities with Number type that is correctly found
* correct_entities_with_Temporal_type = the number of entities with Temporal type that is correctly found
* correct_entities_with_Monetary_Unit_type = the number of entities with Monetary Unit type that is correctly found
* correct_entities_with_Reporting_Entity_type = the number of entities with Reporting Entity type that is correctly found
* correct_entities_with_Financial_Concepts_type = the number of entities with Financial Concepts type that is correctly found
* entity_accuracy = correct_entities / total_entities * 100%
            (entity_accuracy should be a percentage number between 0% and 100%, rounded to two decimal places.)
    If an entity is partially matched (e.g., missing comma or currency symbol), mark it incorrect.

# Step 2: Overall judgment
    Add a 1-2 sentence short justification (overall_explanation) on the LLM outputs.

# Step 3: Output format
Output exactly one valid JSON object:
        {
"total_entities": 0,
"total_entities_with_Number_type": 0,
"total_entities_with_Temporal_type": 0,
“total_entities_with_Monetary_Unit_type”:0,
“total_entities_with_Reporting_Entity_type”:0,
“total_entities_with_Financial_Concepts_type”:0,
"correct_entities": 0,
"correct_entities_with_Number_type": 0,
"correct_entities_with_Temporal_type": 0,
"correct_entities_with_Monetary_Unit_type": 0,
"correct_entities_with_Reporting_Entity_type": 0,
"correct_entities_with_Financial_Concepts_type": 0,
"entity_accuracy": 0.00%,
"overall_explanation": “”
        }

Rules:
* Output only valid JSON (no extra text).
* Numbers must be numeric, not strings.
"""


In [ ]:
def get_user_prompt(GROUND_TRUTH_HTML, MODEL_HTML):
    
    return f"""# Inputs
    Ground truth HTML (with entity tags): {GROUND_TRUTH_HTML}

    Model-generated HTML (no entity tags): {MODEL_HTML}

# Outputs
"""

In [ ]:
def get_response(user_input):
    
    completion = Client.chat.completions.create(
        model="gpt-4o",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": JUDGE_PROMPT_TEMPLATE},
            {"role": "user", "content": user_input}
        ]
    )
    return completion.choices[0].message.content

In [ ]:
def read_html_txt(file_path):

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
    except UnicodeDecodeError:
        with open(file_path, "r", encoding="latin-1") as f:
            content = f.read()
    return content


In [ ]:
def write_case_result(output_path: str, result: list):
    parent_dir = os.path.dirname(output_path)
    os.makedirs(parent_dir, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result,f,indent=4)

    print(f"Results saved to {output_path}")

In [ ]:
def evaluate_performance(result: list):
    total_entity = 0
    total_number_entity = 0
    total_date_entity = 0

    total_correct_entity = 0
    total_correct_number_entity = 0
    total_correct_date_entity = 0

    for line in result:

        entity = line.get("total_entities", 0)
        number_entity = line.get("total_entities_with_Number_type", 0)
        date_entity = line.get("total_entities_with_Date_type", 0)

        correct_entity = line.get("correct_entities", 0)
        correct_number_entity = line.get("correct_entities_with_Number_type", 0)
        correct_date_entity = line.get("correct_entities_with_Date_type", 0)


        total_entity += entity
        total_number_entity += number_entity
        total_date_entity += date_entity

        total_correct_entity += correct_entity
        total_correct_number_entity += correct_number_entity
        total_correct_date_entity += correct_date_entity


    entity_acc = round((total_correct_entity / total_entity) * 100, 2) if total_entity else 0.0
    number_entity_acc = round((total_correct_number_entity / total_number_entity) * 100, 2) if total_number_entity else 0.0
    date_entity_acc = round((total_correct_date_entity / total_date_entity) * 100, 2) if total_date_entity else 0.0

    return {
        "entity_acc": entity_acc,
        "number_entity_acc": number_entity_acc,
        "date_entity_acc": date_entity_acc
    }


In [ ]:
Results = []
for i in tqdm(range(total_samples)):
    GROUND_TRUTH_HTML_PATH = os.path.join(gold_html, f"gold_{i}.txt")
    MODEL_HTML_PATH = os.path.join(model_html, f"pred_{i}.txt")

    GROUND_TRUTH_HTML = read_html_txt(GROUND_TRUTH_HTML_PATH)
    MODEL_HTML = read_html_txt(MODEL_HTML_PATH)

    user_input = get_user_prompt(GROUND_TRUTH_HTML, MODEL_HTML)
    response = get_response(user_input)
    Results.append(json.loads(response))

In [ ]:
Results[0]

In [ ]:
write_case_result(os.path.join(case_result,f"{model_name}.json"), Results)

In [ ]:
evaluate_performance(Results)